# 02 — Matrices

**Objetivo:** Entender matrices como arreglos 2D, sus operaciones y tipos especiales relevantes en analytics (covarianza, funnels, atribución).

## Contexto matemático

Una matriz $A \in \mathbb{R}^{m \times n}$ tiene $m$ filas y $n$ columnas:

$$A = \begin{pmatrix} a_{11} & a_{12} & \cdots & a_{1n} \\
a_{21} & a_{22} & \cdots & a_{2n} \\
\vdots & & \ddots & \vdots \\
a_{m1} & a_{m2} & \cdots & a_{mn} \end{pmatrix}$$

**En analytics:** filas = observaciones (canales, usuarios), columnas = features (etapas del funnel).

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)
print('Setup listo')

## 1 — Matriz funnel: canales × etapas

Cada fila es un canal de adquisición. Cada columna es una etapa del funnel. El elemento $a_{ij}$ es la tasa de conversión del canal $i$ en la etapa $j$.

In [ ]:
# Funnel: 5 canales × 4 etapas
# Etapas: [Visita→Registro, Registro→Onboarding, Onboarding→Activación, Activación→Pago]
canales = ['Organic', 'Paid Search', 'Email', 'Referral', 'Direct']
etapas  = ['Visita→Reg', 'Reg→Onboard', 'Onboard→Act', 'Act→Pago']

funnel = np.array([
    [0.042, 0.61, 0.55, 0.38],   # Organic
    [0.031, 0.48, 0.62, 0.41],   # Paid Search
    [0.078, 0.72, 0.49, 0.35],   # Email
    [0.055, 0.83, 0.71, 0.52],   # Referral
    [0.028, 0.55, 0.58, 0.44],   # Direct
])

print(f'Shape: {funnel.shape}  → {funnel.shape[0]} canales × {funnel.shape[1]} etapas')
print('\nMatriz funnel (tasas de conversión):')
print(funnel)

## 2 — Operaciones elementales y traspuesta

**Operaciones elemento a elemento** (`+`, `-`, `*`, `/` con NumPy): actúan componente a componente.

**Traspuesta** $A^T$: intercambia filas y columnas, $a^T_{ij} = a_{ji}$.

- Si $A$ tiene shape $(m, n)$, entonces $A^T$ tiene shape $(n, m)$.
- Propiedad: $(AB)^T = B^T A^T$.
- Una matriz **simétrica** satisface $A = A^T$.

In [ ]:
# Comparar con el mes anterior (elemento a elemento)
funnel_ant = funnel * (1 + np.random.uniform(-0.15, 0.15, funnel.shape))
delta = funnel - funnel_ant
print('Delta vs mes anterior (puntos porcentuales):')
print(delta.round(3))

# Traspuesta: ahora etapas × canales
print(f'\nShape original: {funnel.shape}')
print(f'Shape traspuesta: {funnel.T.shape}')
print('\nPrimera columna de funnel == primera fila de funnel.T:', np.allclose(funnel[:, 0], funnel.T[0, :]))

## 3 — Matrices especiales

| Tipo | Definición | Uso en analytics |
|---|---|---|
| Identidad $I$ | $I_{ij} = 1$ si $i=j$, 0 si no | Operador neutro |
| Diagonal $D$ | $a_{ij} = 0$ si $i \neq j$ | Escalas por canal |
| Simétrica | $A = A^T$ | Matriz de covarianza |
| Triangular inferior/superior | $a_{ij}=0$ para $j>i$ o $i>j$ | Eliminación gaussiana |
| Cero | $a_{ij}=0\ \forall i,j$ | Inicialización |

In [ ]:
n = 5

I = np.eye(n)
print('Identidad 5×5:')
print(I)

# Diagonal: presupuesto por canal (en miles)
presupuestos = np.array([120, 85, 60, 95, 40])
D = np.diag(presupuestos)
print('\nMatriz diagonal (presupuestos):')
print(D)

# Matriz triangular inferior: dependencias del funnel (etapa j depende de 1..i)
T_low = np.tril(np.ones((4, 4)))
print('\nTriangular inferior (estructura de dependencia del funnel):')
print(T_low)

# Traza
print(f'\nTraza de la identidad np.trace(I) = {np.trace(I)}')

## 4 — Matriz de covarianza como matriz simétrica

La matriz de covarianza de variables $X_1, \ldots, X_p$ es:

$$\Sigma_{ij} = \text{Cov}(X_i, X_j) = \mathbb{E}[(X_i - \mu_i)(X_j - \mu_j)]$$

Propiedades:
- **Simétrica:** $\Sigma = \Sigma^T$ (porque $\text{Cov}(X,Y)=\text{Cov}(Y,X)$).
- **Semidefinida positiva:** $\mathbf{v}^T \Sigma \mathbf{v} \geq 0$ para todo $\mathbf{v}$.
- La diagonal contiene las **varianzas** de cada variable.

In [ ]:
# Generar datos de métricas de 5 canales (n=200 períodos)
np.random.seed(42)
metricas_raw = np.random.multivariate_normal(
    mean=[0]*5,
    cov=[[1, 0.8, 0.3, -0.2, 0.1],
         [0.8, 1, 0.5, -0.1, 0.2],
         [0.3, 0.5, 1,  0.0, 0.4],
         [-0.2,-0.1,0.0, 1, -0.3],
         [0.1, 0.2, 0.4,-0.3, 1]],
    size=200
)

# Covarianza muestral: (X - mu)^T (X - mu) / (n-1)
Sigma = np.cov(metricas_raw.T)
print('Matriz de covarianza (5×5):')
print(Sigma.round(3))
print(f'\n¿Es simétrica? {np.allclose(Sigma, Sigma.T)}')
print(f'Diagonal (varianzas): {np.diag(Sigma).round(3)}')

## 5 — Visualización: heatmap de la matriz funnel y covarianza

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Heatmap funnel
ax = axes[0]
im = ax.imshow(funnel, cmap='Blues', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(etapas))); ax.set_xticklabels(etapas, rotation=20, ha='right')
ax.set_yticks(range(len(canales))); ax.set_yticklabels(canales)
ax.set_title('Matriz Funnel: canales × etapas')
plt.colorbar(im, ax=ax, shrink=0.8)
for i in range(funnel.shape[0]):
    for j in range(funnel.shape[1]):
        ax.text(j, i, f'{funnel[i,j]:.2f}', ha='center', va='center', fontsize=9,
                color='white' if funnel[i,j] > 0.6 else 'black')

# Heatmap covarianza
ax2 = axes[1]
nom = ['C1','C2','C3','C4','C5']
im2 = ax2.imshow(Sigma, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
ax2.set_xticks(range(5)); ax2.set_xticklabels(nom)
ax2.set_yticks(range(5)); ax2.set_yticklabels(nom)
ax2.set_title('Matriz de Covarianza (simétrica)')
plt.colorbar(im2, ax=ax2, shrink=0.8)
for i in range(5):
    for j in range(5):
        ax2.text(j, i, f'{Sigma[i,j]:.2f}', ha='center', va='center', fontsize=8,
                 color='white' if abs(Sigma[i,j]) > 0.6 else 'black')

plt.tight_layout()
plt.show()

## Resumen

| Concepto | Descripción | NumPy |
|---|---|---|
| Traspuesta | Intercambia filas/columnas | `A.T` |
| Identidad | Neutro multiplicativo | `np.eye(n)` |
| Diagonal | Solo diagonal no nula | `np.diag(v)` |
| Triangular | Cero arriba o abajo de la diagonal | `np.tril(A)`, `np.triu(A)` |
| Traza | Suma de la diagonal | `np.trace(A)` |
| Covarianza | Simétrica, semidefinida positiva | `np.cov(X.T)` |

**Siguiente:** `03_systems_of_equations.ipynb`